# Web Scraping & Product Price Analysis – Air Conditioners (Amazon India)
**Stage 2 – Data Acquisition & Filtering** (simplified, beginner version)

Pipeline position: **2 of 8**

## Objective
1. (Optional) **Scrape** AC listings from Amazon India with `requests` + `BeautifulSoup`.
2. **Filter** the raw listings: keep only real air conditioners, remove duplicates, and look at useful subsets.

## Step 1 – Setup

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Folder that holds all the CSV files of the project.
# (In Google Colab: upload the CSV files and change this to "/content/")
DATA = "../0. DataSet/"

print("Setup complete.")

Setup complete.


## Step 2 – Scraper (optional)
Set `RUN_SCRAPER = True` to scrape fresh data. Keep it `False` to simply load the CSV we already have (Amazon often blocks scrapers, so the saved file is the safe choice).

How the scraper works, in plain words:
1. `requests.get(url)` downloads a search page.
2. `BeautifulSoup` turns the HTML into something we can search.
3. `soup.select(...)` finds every product card.
4. For each card we pick out the name, price, rating etc. and add them to a list.
5. The list is turned into a DataFrame and saved as a CSV.

In [6]:
import os
import pandas as pd

RUN_SCRAPER = False
RAW_FILE = "Air_Conditioners_Raw.csv"

# Fallback check for Google Colab environment
if os.path.exists("/content/" + RAW_FILE):
    OUTPUT_FILE = "/content/" + RAW_FILE
else:
    OUTPUT_FILE = DATA + RAW_FILE

MAX_PAGES = 5            # number of result pages to scrape (increase for more data)

if RUN_SCRAPER:
    import time
    import random
    import requests
    from bs4 import BeautifulSoup

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36",
               "Accept-Language": "en-IN,en;q=0.9"}

    products = []            # every product found will be added to this list

    for page in range(1, MAX_PAGES + 1):
        print("Scraping page", page)
        url = "https://www.amazon.in/s?k=air+conditioner&page=" + str(page)
        response = requests.get(url, headers=headers, timeout=20)

        if response.status_code != 200:
            print("Page not available (status", response.status_code, "). Stopping.")
            break

        soup = BeautifulSoup(response.text, "html.parser")
        cards = soup.select("div[data-component-type='s-search-result']")
        print("  Listings found:", len(cards))
        if len(cards) == 0:
            break

        for card in cards:
            # --- name and link ---
            title = card.find("h2")
            if title is None:
                continue                      # skip cards without a title
            name = title.get_text(" ", strip=True)
            link_tag = card.find("a", href=True)
            link = "https://www.amazon.in" + link_tag["href"]

            # --- image ---
            img_tag = card.find("img", class_="s-image")
            image = img_tag["src"] if img_tag else None

            # --- rating, e.g. "4.2 out of 5 stars" ---
            ratings = None
            rating_tag = card.select_one("span.a-icon-alt")
            if rating_tag:
                ratings = float(rating_tag.get_text().split(" ")[0])

            # --- number of ratings, e.g. "2,255" ---
            no_of_ratings = None
            count_tag = card.select_one("span.a-size-base.s-underline-text")
            if count_tag:
                text = count_tag.get_text(strip=True).replace(",", "").replace("(", "").replace(")", "")
                if text.isdigit():
                    no_of_ratings = float(text)

            # --- selling price (big price), e.g. "₹32,999" ---
            discount_price = None
            price_tag = card.select_one("span.a-price:not(.a-text-price) span.a-offscreen")
            if price_tag:
                discount_price = float(price_tag.get_text().replace("₹", "").replace(",", ""))

            # --- MRP (struck-through price) ---
            actual_price = None
            mrp_tag = card.select_one("span.a-price.a-text-price span.a-offscreen")
            if mrp_tag:
                actual_price = float(mrp_tag.get_text().replace("₹", "").replace(",", ""))

            products.append({"name": name, "main_category": "appliances", "sub_category": "Air Conditioners",
                             "image": image, "link": link, "ratings": ratings, "no_of_ratings": no_of_ratings,
                             "discount_price": discount_price, "actual_price": actual_price})

        time.sleep(random.randint(3, 7))      # wait a few seconds so we do not overload the website

    scraped = pd.DataFrame(products)
    scraped = scraped.drop_duplicates(subset="link")      # sponsored ads repeat the same product
    scraped.to_csv(OUTPUT_FILE, index=False)
    print("Saved", len(scraped), "listings to", OUTPUT_FILE)

raw_df = pd.read_csv(OUTPUT_FILE)
print("Raw listings available for filtering:", raw_df.shape)

Raw listings available for filtering: (32, 9)


## Step 3 – Start filtering
We work on a copy so the raw data stays safe, and keep a small list to record how many rows remain after each step.

In [7]:
df = raw_df.copy()
rows_left = [("Raw listings", len(df))]
df.head(3)

,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price
0,Helium Smart Choice Air Smart Inverter Split A...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51diNudbdO...,https://www.amazon.in/Helium-Inverter-Capacity...,3.3,212.0,22990.0,40100.0
1,IFB 0.95 Ton 3 Star 2026 AI Powered Inverter S...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51M-MqPqBr...,https://www.amazon.in/IFB-Conditioner-Compress...,4.4,200.0,30990.0,50990.0
2,"Blue Star 1.5 Ton 3 Star, New Star Rated, Inve...",appliances,Air Conditioners,https://m.media-amazon.com/images/I/61-yR4Rid4...,https://www.amazon.in/Blue-Star-Inverter-Comfo...,4.9,12.0,34990.0,17495.0


## Step 4 – Keep only air conditioners
Search results can include accessories (stands, covers, remotes ...). We look for AC words in the product name using `str.contains()`.
`|` means **or**.  `~` means **not**.

In [8]:
name_lower = df["name"].str.lower()

has_ac_word = name_lower.str.contains("air conditioner|split|window|cassette|tower| ac ")
has_ton = name_lower.str.contains("ton")
is_accessory = name_lower.str.contains("stand|bracket|cover|stabilizer|remote|cooler|pipe|tray|cleaner|spray|cable|kit")

# keep a listing if it looks like an AC, and is not an accessory (unless it states a tonnage)
is_ac = has_ac_word & (~is_accessory | has_ton)

print("Removed as non-AC:", (~is_ac).sum())
print(df.loc[~is_ac, "name"].head(10))

df = df[is_ac]
rows_left.append(("After AC-only filter", len(df)))

Removed as non-AC: 0
Series([], Name: name, dtype: object)


## Step 5 – Remove duplicates
Every Amazon product has a 10-character id (ASIN) inside its link, after `/dp/`. We cut it out of the link with `split()` and drop repeated ids.

In [9]:
df["asin"] = df["link"].str.split("/dp/").str[1].str[:10]

df = df.dropna(subset=["asin", "name"])
rows_left.append(("After valid link / name", len(df)))

print("Duplicate products:", df["asin"].duplicated().sum())
df = df.drop_duplicates(subset="asin")
rows_left.append(("After removing duplicates", len(df)))

Duplicate products: 0


## Step 6 – Useful subsets (filtering with conditions)
Each condition gives `True/False` for every row; `&` means **and**.

In [10]:
has_price = df["discount_price"].notna()
well_rated = df["ratings"] >= 4.0
popular = df["no_of_ratings"] >= df["no_of_ratings"].median()
mid_range = df["discount_price"].between(25000, 40000)

print("Listings with a selling price  :", has_price.sum())
print("Rated 4.0 or above             :", well_rated.sum())
print("Above-median number of reviews :", popular.sum())
print("Price between 25,000 and 40,000:", mid_range.sum())
print("Rated 4+ AND price 25k-40k     :", (well_rated & mid_range).sum())

Listings with a selling price  : 31
Rated 4.0 or above             : 20
Above-median number of reviews : 16
Price between 25,000 and 40,000: 22
Rated 4+ AND price 25k-40k     : 12


## Step 7 – Highest and lowest priced listings

In [11]:
columns = ["name", "ratings", "no_of_ratings", "discount_price"]

print("5 highest-priced listings")
print(df.sort_values("discount_price", ascending=False)[columns].head(5))
print()
print("5 lowest-priced listings")
print(df.sort_values("discount_price")[columns].head(5))

5 highest-priced listings
                                                 name  ratings  no_of_ratings  \
30  Blue Star 2 Ton 3 Star, New Star Rated, Invert...      4.6           64.0   
17  Daikin 1.5 Ton 5 Star Inverter Split AC (Coppe...      4.1          218.0   
24  LG 5800W Max Cooling Capacity, 5 Star, 100% Co...      4.0          879.0   
27  Hitachi Smart Choice 1.5 Ton 5 Star, New Star ...      4.1          260.0   
25  Samsung 1.5 Ton 4 Star Bespoke AI WindFree Inv...      4.5           33.0   

    discount_price  
30         49804.0  
17         48990.0  
24         48490.0  
27         44999.0  
25         42490.0  

5 lowest-priced listings
                                                 name  ratings  no_of_ratings  \
12  Oakter 0.5 Ton Inverter Studio AC 5000 (2026 M...      4.0          192.0   
0   Helium Smart Choice Air Smart Inverter Split A...      3.3          212.0   
5   Sharp 0.95 Ton 3 star, New Star Rated,Split AC...      4.0          133.0   
6   LG Esse

## Step 8 – Save the filtered data

In [13]:
import os

summary = pd.DataFrame(rows_left, columns=["step", "rows"])
print(summary)

filtered_df = df.drop(columns="asin")

# Fallback check for Google Colab environment
if not os.path.exists(DATA):
    OUTPUT_PATH = "/content/Air_Conditioners_Filtered.csv"
else:
    OUTPUT_PATH = DATA + "Air_Conditioners_Filtered.csv"

filtered_df.to_csv(OUTPUT_PATH, index=False)
print("Saved:", filtered_df.shape, "to", OUTPUT_PATH)

                        step  rows
0               Raw listings    32
1       After AC-only filter    32
2    After valid link / name    32
3  After removing duplicates    32
Saved: (32, 9) to /content/Air_Conditioners_Filtered.csv


**Output of Stage 2:** `Air_Conditioners_Filtered.csv`.  **Next → Stage 3: Data Extraction.**